# nano Beatrix — the find screen's cores: Alhena-F, a request that must find its record

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_find_cores.ipynb)

**The question**: on the scale test's grid cores, independently trained members merged under the product gate are exact when the gate is given the true address, and not when the request is the core's own: Alhena writes its index about .90 / .81 right, and its address head, trained at the questions, names whatever the question's address stream carries. Here the same head is ALSO trained at each index entry's SEP, where both address streams carry the null anchor and nothing earlier in the row names the entry key's address except its record: to name the address there, the core must find the record by its key.

| part | what | why |
|---|---|---|
| READ 0 | the scale test's four stored grid cores (Castor and Alhena, seeds 1 and 2, 24,000 steps) with their stored members, read with no training: the scale test's form again (its cells re-read) and the two-pass form below (each core's own index bytes in the row) | before any claim about a stored core's head: every index read of the scale test was teacher-forced |
| Alhena-F s1, s2 | Alhena (`alhena-both`: Castor's layout turned by the anchored multi-position rotary), the scale test's recipe at 24,000 steps, plus the head's term at the index entries | the request that must find its record |
| Alhena s1 again | the scale test's Alhena seed 1 at 24,000 steps verbatim (series `scalerep`) | the replication arm: Alhena's same-seed repeat, never measured |
| the retrain rule | a seed whose Alhena-F misses the find at 24,000 steps trains once more at 48,000 steps | registered before any run |

**The loss** (one pure Adam, one clip): L = (1/N) sum_t CE + 1 x (1/Q) sum_q BCE(head at the question's SEP) + 1 x (1/E) sum_e BCE(head at the index entry's SEP); the head is one Linear(256 -> 16) on the final norm's state (two digit levels x eight digits), its bias born at logit(1/8). Each term passes its own novelty gate at step 0 (1 - |cos| between its gradient and the CE's over the shared trunk >= .05) and records its descent.

**The two-pass read**: pass 1 reads the head at each index entry with every question at the null anchor (the index precedes the questions); a digit enters the question's stream when its own score passes .5 (exactly one digit a level, else the null anchor: no digit is compared with another); pass 2 reads the answers with that address in the stream, and the gate takes the head's scores. The free-running index is the fixed point of feeding the core its own index bytes (the sequential greedy writing).

**The cells**: C1 pins the trainer and holds the decision block; C2 builds the data, prints the plan, runs 200 steps of Alhena-F on this card for the price line (a projection past 1.5x the planned run stops the session for a re-plan) and probes the push path; C3a runs READ 0; C3 trains the parts in order; C3b runs the retrain rule; C4 renders the tables. Every finished run ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/` with the head and `scale.json`; READ 0 under `bed/reports/read0/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The trainer and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.72"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 72), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the stored cores come from here; every finished run ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000)                     # the formation trace every 2,000 steps
FIND = dict(head_at='both', head_lam_i=1.0)            # THE NEW TERM: the same head at each index entry's SEP, weight 1
PARTS = [                                              # the gate order: the two find cores gate the members and the retrain rule; then the repeat
    ('Alhena-F s1', dict(BED, **GRID_ROWS, **FIND, arms='alhena-both', seeds='1', series='find')),
    ('Alhena-F s2', dict(BED, **GRID_ROWS, **FIND, arms='alhena-both', seeds='2', series='find')),
    ('Alhena s1 again', dict(BED, **GRID_ROWS, arms='alhena-both', seeds='1', series='scalerep')),   # the replication arm's repeat
]
STORED_TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_ahdigit1_scale'
STORED = [(f'{arm}_s{seed}{STORED_TAIL}', seed) for seed in (1, 2) for arm in ('castor-both', 'alhena-both')]   # READ 0's cores
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the scale test's read and calibration rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of Alhena-F on this card for the price line; its record stays local
PLAN_MIN = 16.5                                        # the planned minutes of one 24,000-step Alhena run on this card (978-980 s measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', len(PARTS), 'parts,', len(STORED), 'stored cores for READ 0 | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena-F on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo = 0
for name, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']
smoke = dict(PARTS[0][1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_run <= STOP_FACTOR * PLAN_MIN
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.1f} min (planned "
      f"{PLAN_MIN} min); the {todo} runs left about {per_run * todo / 60:.1f} h (+ a few minutes of end reads a run)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_find.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [n for n, _ in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano find cores: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

In [ ]:
# C3a (R) — READ 0: the four stored grid cores with their stored members, no training (fp32, TF32 off): the scale test's form again (its
# cells re-read against the stored record) and the two-pass form (the free-running index; the head's sources). ~10 min; each record ships.
import time
from geolip.nano_beatrix import rotary_scale as rs
os.makedirs(f'{OUT}/read0', exist_ok=True)
ship0 = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
t_all = time.time()
for n, (tag, seed) in enumerate(STORED, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: READ 0 stops here')
        break
    t0 = time.time()
    cf, mf = rs.fetch_stored(REPO, tag, f'{tag}_members_l16000_m32000', f'{OUT}/hub')
    rec = rs.read0(cf, mf, f'{OUT}/read0/READ0_{tag}.json', seed, HELD, read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES)
    d = rec['find']['diag']
    print(f"READ 0 {n}/{len(STORED)} {tag[:22]}: the index written teacher-forced {d['written_index_right']:.4f}, free-running "
          f"{d['indexf_right']:.4f} (passes mean {d['passes']['mean']:.1f}); continuity {rec.get('continuity')}; "
          f"{(time.time() - t0) / 60:.1f} min, {(time.time() - t_all) / 60:.1f} min spent, {len(STORED) - n} cores left")
    torch.cuda.empty_cache()
    if ship0:
        ship0.submit(f'{OUT}/read0', 'bed/reports/read0')
if ship0:
    print('READ 0 shipped' if ship0.drain() else 'READ 0 ships still retrying in the background')

In [ ]:
# C3 (T) — THE SESSION: the parts in gate order, one run at a time, resume-first (runner_done.txt in OUT); a STOP file in OUT starts no
# new run; every finished run ships to REPO as it lands (one background shipper: the card never waits on the Hub).
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = bed_runner._Shipper(push, OUT, print) if push else None
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C3b (T) — THE REGISTERED RETRAIN (written before any run): a seed whose Alhena-F missed THE FIND at 24,000 steps - the head at the index
# entries, on the free-running row, right by its .5 decode on all but at most 2 of the read's entries, its mean truth score >= .99, and the
# 99.9th percentile of its largest wrong digit score on the calibration rows <= .01 at both levels (the core's own end read) - trains once
# more at 48,000 steps before any member trains on it. This cell reads C3's records and runs only what the rule asks (run it after C3).
import json
from geolip.nano_beatrix import bed_runner
TAIL_F = '_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_ahdigit1i1_find'

def _find_read(seed, steps):
    p = f'{OUT}/bed/bed_alhena-both_s{seed}_L{steps}{TAIL_F}.json'
    return json.load(open(p, encoding='utf-8'))['grid']['read']['free_running'] if os.path.exists(p) else None

RETRAIN = []
for seed in (1, 2):
    fr = _find_read(seed, 24000)
    if fr is None:
        print(f'seed {seed}: no 24,000-step Alhena-F record in {OUT}/bed; nothing decided for this seed')
        continue
    h, tau = fr['headif'], fr['calib']['tau']['0.999']
    wrong = round((1 - h['right']) * h['n_entries'])
    met = wrong <= 2 and h['truth'] >= 0.99 and max(tau) <= 0.01
    print(f"seed {seed}: the find {h['right']:.5f} right ({wrong} of {h['n_entries']} wrong), truth {h['truth']:.4f}, calibration Q999 "
          f"{tau} -> {'met' if met else 'MISSED: the 48,000-step retrain runs'}")
    if not met:
        RETRAIN.append((f'Alhena-F s{seed} 48k', dict(PARTS[seed - 1][1], steps=48000)))
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
done2 = []
for name, spec in RETRAIN:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done2.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('retrain session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done2] or 'nothing to retrain')

In [ ]:
# C4 — THE TABLES. The cores: held-out bits, the frame copy, the index (teacher-forced and free-running), the find (the head at the index
# entries on the free-running row), the answers by request, the terms' descent. READ 0: the index both ways, the stored head against the
# free-running index, the marks at the nine rest cells, the continuity with the stored cells. Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | find right / truth / Q999 | answers true / via index / via find | descent q / i | min |',
        '|---|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_alhena-both_*_L*_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    if not r.get('grid'):
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g, hd = r['grid']['read'], r['grid']['head'] or {}
    fr = g.get('free_running') or {}
    hf = fr.get('headif')
    find = f"{hf['right']:.5f} / {hf['truth']:.4f} / {max(fr['calib']['tau']['0.999']):.2e}" if hf else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f} / " + (f"{hf['whole']:.4f}" if hf else '-')
    desc = f"{hd.get('descended')} / {(hd.get('index') or {}).get('descended', '-')}"
    rows.append(f"| {os.path.basename(p)[4:-5][:60]} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / "
                f"{fr.get('indexf_right', float('nan')):.4f} | {find} | {via} | {desc} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| stored core | index TF / FR | head at the question agrees with FR index | all marks, 9 rest cells (arms): INDEXF / HEADPF / HEADIF | continuity marks equal |',
        '|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/read0/READ0_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    d, c = r['find']['diag'], r['find']['stacks']['arms']['cells']
    nine = lambda s: (sum(all(c[f'{s}_U{u}_REST_m{m}'][k] for k in ('PROTECTED', 'QUIET', 'EXACT')) for u in (5, 8, 10) for m in (16, 32, 48))
                      if f'{s}_U5_REST_m16' in c else '-')
    agree = d['sources'].get('HEADPF', {}).get('argmax_agrees_with_indexf')
    cont = r.get('continuity') or {}
    rows.append(f"| {r['core'][:24]} | {d['written_index_right']:.4f} / {d['indexf_right']:.4f} | {'-' if agree is None else f'{agree:.4f}'} | "
                f"{nine('INDEXF')} / {nine('HEADPF')} / {nine('HEADIF')} | {cont.get('marks_equal')} of {cont.get('cells')} |")
display(Markdown('\n'.join(rows)))